In [0]:
VOLUME_PATH  = "/Volumes/medical_catalog/medical/raw_files/medical_transcriptions.csv"
BRONZE_TABLE = "medical_catalog.bronze.medical_transcriptions"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("inferSchema", "false") \
    .load(VOLUME_PATH)

print("Rows:", df_raw.count())
print("Columns:", df_raw.columns)
display(df_raw.limit(5))

In [0]:
df_ingested = df_raw.toDF(
    "no",
    "description",
    "medical_specialty",
    "sample_name",
    "transcription",
    "keywords"
) \
.withColumn("ingested_at",   current_timestamp()) \
.withColumn("source_file",   lit("medical_transcriptions.csv")) \
.withColumn("source_volume", lit("/Volumes/medical_catalog/medical/raw_files/")) \
.withColumn("batch_id",      lit("batch_001"))

display(df_ingested.limit(5))

In [0]:
df_ingested.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

print("Ingestion complete!")
print("Row count:", spark.table(BRONZE_TABLE).count())